In [11]:

import re
import os
import json
import time
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm
from SPARQLWrapper import SPARQLWrapper, JSON



# Hide warnings for cleaner output
import warnings
warnings.filterwarnings('ignore')

def parse_nt_to_dataframe(filepath: str) -> pd.DataFrame:
    """Parses the N-Triples dataset and returns a clean Pandas DataFrame."""
    NS_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#type>"
    NS_SUBJECT = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#subject>"
    NS_PREDICATE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#predicate>"
    NS_OBJECT = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#object>"
    NS_TRUTH = "<http://swc2017.aksw.org/hasTruthValue>"
    STATEMENT_TYPE = "<http://www.w3.org/1999/02/22-rdf-syntax-ns#Statement>"

    raw_data = {}
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('#'): continue
            
            parts = line.split(' ', 2)
            if len(parts) < 3: continue
                
            s, p, obj_part = parts[0], parts[1], parts[2]
            
            # Safely remove the trailing dot
            o = obj_part[:-2] if obj_part.endswith(' .') else obj_part.rstrip('\n')
            raw_data.setdefault(s, {})[p] = o

    facts_list = []
    for uri_bracketed, props in raw_data.items():
        if props.get(NS_TYPE) != STATEMENT_TYPE: continue
            
        uri = uri_bracketed.strip('<>')
        truth = None
        if NS_TRUTH in props:
            val = props[NS_TRUTH]
            if val.startswith('"'):
                try: truth = float(val.split('"')[1])
                except ValueError: pass

        facts_list.append({
            "fact_uri": uri,
            "subject": props.get(NS_SUBJECT, '').strip('<>'),
            "predicate": props.get(NS_PREDICATE, '').strip('<>'),
            "object": props.get(NS_OBJECT, '').strip('<>'),
            "truth_value": truth
        })

    return pd.DataFrame(facts_list)

# Load the training data
df_train = parse_nt_to_dataframe("KG-2022-train.nt")
print(f"Loaded {len(df_train)} training facts.")
display(df_train.head(3))

Loaded 1234 training facts.


,fact_uri,subject,predicate,object,truth_value
0,http://swc2017.aksw.org/task2/dataset/3226691,http://dbpedia.org/resource/David_Lee_(basketb...,http://dbpedia.org/ontology/team,http://dbpedia.org/resource/Houston_Rockets,0.0
1,http://swc2017.aksw.org/task2/dataset/3320759,http://dbpedia.org/resource/Nenad_Zimonjić,http://dbpedia.org/ontology/award,http://dbpedia.org/resource/Belgrade,0.0
2,http://swc2017.aksw.org/task2/dataset/3642843,http://dbpedia.org/resource/Om_Shanti_Om,http://dbpedia.org/ontology/starring,http://dbpedia.org/resource/Uma_Thurman,0.0


In [ ]:
import requests
import urllib3

# Completely suppress SSL certificate warnings
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

class GraphFeatureExtractor:
    def __init__(self, endpoint="https://dbpedia.org/sparql", cache_file="sparql_cache.json"):
        self.endpoint = endpoint
        self.cache_file = cache_file
        self.cache = self._load_cache()

    def _load_cache(self):
        if os.path.exists(self.cache_file):
            with open(self.cache_file, 'r') as f:
                return json.load(f)
        return {}

    def _save_cache(self):
        with open(self.cache_file, 'w') as f:
            json.dump(self.cache, f)

    def _ask_query(self, query: str) -> int:
        """Executes an ASK query using the requests library with a hard SSL bypass."""
        if query in self.cache:
            return self.cache[query]

        try:
            time.sleep(0.1) 
            response = requests.get(
                self.endpoint,
                params={"query": query, "format": "json"},
                verify=False, 
                timeout=15
            )
            response.raise_for_status()
            data = response.json()
            ans = 1 if data.get("boolean", False) else 0
            self.cache[query] = ans
            return ans
        except Exception as e:
            print(f"Query failed: {e}")
            return 0 

    def extract_features(self, subject: str, predicate: str, obj: str):
        """Extracts the core 4 graph features for an ML model."""
        features = {}
        
        features['direct_match'] = self._ask_query(f"ASK {{ <{subject}> <{predicate}> <{obj}> }}")
        
        features['inverse_match'] = self._ask_query(f"ASK {{ <{obj}> <{predicate}> <{subject}> }}")
        
        features['subject_exists'] = self._ask_query(f"ASK {{ {{ <{subject}> ?p ?o }} UNION {{ ?s ?p <{subject}> }} }}")
        
        features['object_exists'] = self._ask_query(f"ASK {{ {{ <{obj}> ?p ?o }} UNION {{ ?s ?p <{obj}> }} }}")
        
        return features

extractor = GraphFeatureExtractor()
test_features = extractor.extract_features(
    subject=df_train.iloc[0]['subject'], 
    predicate=df_train.iloc[0]['predicate'], 
    obj=df_train.iloc[0]['object']
)


extractor._save_cache()

print("Features for fact 0:", test_features)

Features for fact 0: {'direct_match': 0, 'inverse_match': 0, 'subject_exists': 1, 'object_exists': 1}


In [ ]:

from tqdm import tqdm # Use standard text progress bar to bypass ipywidgets error

print("Calculating Predicate Priors...")
predicate_priors = df_train.groupby('predicate')['truth_value'].mean().to_dict()
global_mean_prior = df_train['truth_value'].mean()

print("Extracting Graph Features (This will take ~2 minutes on the first run)...")
feature_rows = []

# Loop through all facts with a text-based progress bar
for idx, row in tqdm(df_train.iterrows(), total=len(df_train)):
    graph_features = extractor.extract_features(row['subject'], row['predicate'], row['object'])
    
    prior = predicate_priors.get(row['predicate'], global_mean_prior)
    
    # 3. Combine them into a single row
    feature_row = {
        'fact_uri': row['fact_uri'],
        'direct_match': graph_features['direct_match'],
        'inverse_match': graph_features['inverse_match'],
        'subject_exists': graph_features['subject_exists'],
        'object_exists': graph_features['object_exists'],
        'predicate_prior': prior,
        'truth_value': row['truth_value'] # The target variable (y)
    }
    feature_rows.append(feature_row)

# Save the cache immediately after finishing the loop
extractor._save_cache()

# Create the final feature DataFrame
df_features = pd.DataFrame(feature_rows)
print("\nFeature Matrix Generation Complete!")
display(df_features.head())

Calculating Predicate Priors...
Extracting Graph Features (This will take ~2 minutes on the first run)...


100%|██████████| 1234/1234 [03:37<00:00,  5.66it/s]


Feature Matrix Generation Complete!


,fact_uri,direct_match,inverse_match,subject_exists,object_exists,predicate_prior,truth_value
0,http://swc2017.aksw.org/task2/dataset/3226691,0,0,1,1,0.513699,0.0
1,http://swc2017.aksw.org/task2/dataset/3320759,0,0,1,1,0.496689,0.0
2,http://swc2017.aksw.org/task2/dataset/3642843,0,0,1,1,0.506757,0.0
3,http://swc2017.aksw.org/task2/dataset/3800661,1,0,1,1,0.661458,1.0
4,http://swc2017.aksw.org/task2/dataset/3386366,1,0,1,1,0.496689,1.0


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


feature_cols = ['direct_match', 'inverse_match', 'subject_exists', 'object_exists', 'predicate_prior']
X = df_features[feature_cols]
y = df_features['truth_value']

# Build the Model Pipeline
model = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', random_state=42))

cv_auc_scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')

print("=== Local Model Evaluation ===")
print(f"Fold AUC Scores: {cv_auc_scores}")
print(f"Average Expected ROC AUC: {cv_auc_scores.mean():.4f}")

if cv_auc_scores.mean() > 0.60:
    print("STATUS: PASSING! Your model is scoring above the 60% threshold.")
else:
    print("STATUS: NEEDS IMPROVEMENT. We may need to add more features.")

# Train the final model 
final_model = model.fit(X, y)
print("Final model trained and ready for Inference!")

classifier = final_model.named_steps['logisticregression']
importances = pd.DataFrame({
    'Feature': feature_cols,
    'Importance/Weight': classifier.coef_[0]
}).sort_values(by='Importance/Weight', ascending=False)

print("\nModel Feature Importances:")
display(importances)

Exception ignored while calling deallocator <function tqdm.__del__ at 0x11433e4b0>:
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/std.py", line 1148, in __del__
    self.close()
  File "/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/notebook.py", line 279, in close
    self.disp(bar_style='danger', check_delay=False)
AttributeError: 'tqdm_notebook' object has no attribute 'disp'


=== Local Model Evaluation ===
Fold AUC Scores: [0.75582011 0.83409392 0.78412698 0.77728175 0.75578912]
Average Expected ROC AUC: 0.7814
✅ STATUS: PASSING! Your model is scoring above the 60% threshold.
Final model trained and ready for Inference!

Model Feature Importances:


,Feature,Importance/Weight
0,direct_match,2.779246
1,inverse_match,0.643039
4,predicate_prior,0.169371
3,object_exists,0.000000
2,subject_exists,-0.069535


In [ ]:

df_test = parse_nt_to_dataframe("/Users/ankithbharadwaj/University/FOKG/Mini-project/KG-2022-train.nt") 

print(f"Generating predictions for {len(df_test)} test facts...")

test_feature_rows = []
for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    graph_features = extractor.extract_features(row['subject'], row['predicate'], row['object'])
    prior = predicate_priors.get(row['predicate'], global_mean_prior)
    
    test_feature_rows.append({
        'direct_match': graph_features['direct_match'],
        'inverse_match': graph_features['inverse_match'],
        'subject_exists': graph_features['subject_exists'],
        'object_exists': graph_features['object_exists'],
        'predicate_prior': prior
    })

df_test_features = pd.DataFrame(test_feature_rows)

# Using the model trained to predict probabilities
test_probs = final_model.predict_proba(df_test_features)[:, 1]

with open("result.ttl", "w") as out:
    for uri, prob in zip(df_test['fact_uri'], test_probs):
        out.write(f'<{uri}> <http://swc2017.aksw.org/hasTruthValue> "{prob:.6f}"^^<http://www.w3.org/2001/XMLSchema#double> .\n')

print("Success! Created 'result.ttl' in your notebook directory.")

Generating predictions for 1234 test facts...


100%|██████████| 1234/1234 [00:00<00:00, 64392.08it/s]

Success! Created 'result.ttl' in your notebook directory.
